# Notebook 01: Chuẩn bị Dữ liệu, BPE Tokenizer & PhoBERT Token Cache
**Dự án:** Vietnamese ASR → English Translation trên Kaggle T4 ×2 (Dual-GPU DDP)

### Mục tiêu:
1. Kiểm tra môi trường Kaggle (Python >= 3.10, PyTorch, GPU T4).
2. Giải nén mã nguồn dự án `vi_asr_project.zip` vào `/kaggle/working/vi_asr_project`.
3. Tải/Stream tập dữ liệu `nguyendv02/ViMD_Dataset` (revision: `3a5b3015...`), trích xuất manifests train/valid/test chuẩn.
4. Huấn luyện SentencePiece BPE tokenizer (~1,024 vocab) **chỉ trên tập train**.
5. Tạo PhoBERT offline token cache (`[N, 768]` float16 sharded npy) với sliding-window non-truncating stitching.
6. Kiểm toán dữ liệu và rà soát data leakage (Gate G1).

In [ ]:
# 1. Chẩn đoán môi trường Kaggle
!nvidia-smi
!python -c "import torch; print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| Device count:', torch.cuda.device_count())"

In [ ]:
# 2. Nạp mã nguồn vi_asr_project (từ Zip hoặc Git Clone) & Cài đặt dependencies
%cd /kaggle/working
# Cấu hình thư mục cache ghi vào /kaggle/working/.cache để không làm đầy phân vùng root /tmp (~5GB)
%env HF_HOME=/kaggle/working/.cache/huggingface
%env HF_DATASETS_CACHE=/kaggle/working/.cache/huggingface/datasets
%env TRANSFORMERS_CACHE=/kaggle/working/.cache/huggingface/hub
%env TORCH_HOME=/kaggle/working/.cache/torch
%env PYTHONPATH=/kaggle/working/vi_asr_project/src
import os, sys, zipfile
from pathlib import Path

work_dir = Path("/kaggle/working/vi_asr_project")
zip_source = Path("/kaggle/input/vi-asr-code/vi_asr_project.zip")
if not zip_source.exists():
    zip_source = Path("vi_asr_project.zip")

if not (work_dir / "pyproject.toml").exists():
    if zip_source.exists():
        with zipfile.ZipFile(zip_source, 'r') as zf:
            zf.extractall("/kaggle/working")
        print("Unpacked vi_asr_project from zip successfully.")
    else:
        print("Cloning repository from GitHub...")
        !rm -rf /kaggle/working/vi_asr_project
        !git clone --recurse-submodules https://github.com/DongTienDinh/vi-asr-translate.git /kaggle/working/vi_asr_project

%cd /kaggle/working/vi_asr_project
!pip install -q -r requirements-kaggle.in
!pip install -q -e .

src_dir = str(work_dir / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import vi_asr
print('vi_asr successfully loaded from:', vi_asr.__file__)


In [ ]:
# 3. Chuẩn bị manifests dữ liệu từ Hugging Face (mặc định streaming để tránh tràn đĩa)
!python -m vi_asr prepare-data \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml


In [ ]:
# 4. Huấn luyện SentencePiece BPE Tokenizer (Train-only)
!python -m vi_asr build-tokenizer \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml


In [ ]:
# 5. Sinh PhoBERT Offline Token Cache cho tập Train
!python -m vi_asr cache-teacher \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml \
    --split train \
    --device cuda:0


In [ ]:
# 6. Kiểm toán toàn diện phân chia dữ liệu và rà soát Leakage (Gate G1)
!python -m vi_asr audit \
    --config configs/base.yaml \
    --paths configs/paths/kaggle_prepare.yaml \
    --report-dir artifacts/audit_report


In [ ]:
# 7. Đóng gói Audio thành file archive duy nhất (Bảo vệ dữ liệu không bị giới hạn 500 file của Kaggle)
!tar -czf /kaggle/working/artifacts/prepared_audio.tar.gz -C /kaggle/working/artifacts/prepared audio
print('Đã đóng gói prepared_audio.tar.gz thành công để đính kèm sang Notebook 02!')
